# 第一章：金融市场是什么？ / Chapter 1: What Is a Financial Market?

## 你将学到 / What You Will Learn
- 什么是"投资"和"投机" / What "investing" and "speculation" mean
- 现货交易 vs 衍生品交易的区别 / Spot trading vs. derivatives trading
- 什么是"杠杆"，它为什么既是武器也是炸弹 / What "leverage" is and why it is both a weapon and a bomb
- 金融衍生品有哪几大类 / The main families of financial derivatives

---

## 1.1 从买苹果说起 / 1.1 Let's Start With Buying Apples

想象你去水果店：

Imagine you walk into a fruit shop:

> 你花 10 块钱买了 1 斤苹果，拿回家吃了。
>
> You spend 10 yuan on a pound of apples, take them home, and eat them.

这就是**现货交易** —— 你付了钱，拿到了实物，交易结束。

That is **spot trading** — you paid, you received the physical goods, the trade is done.

现在换一种玩法：

Now let's change the rules:

> 你和朋友打赌："下周苹果涨到 12 块，你就给我 2 块差价；跌到 8 块，我给你 2 块。"
>
> You bet a friend: "If apples go to 12 next week, you pay me the 2-yuan difference. If they drop to 8, I pay you 2."

你根本没有买苹果，只是在赌价格涨跌。这就是**衍生品交易**的核心思想 —— 不持有实物，只交易价格变动。

You never actually bought the apples — you only bet on the price move. That is the core idea of **derivatives trading**: you do not hold the underlying asset, you only trade the change in its price.

## 1.2 对比表 / 1.2 Side-by-Side Comparison

|  | 现货交易 / Spot | 衍生品交易 / Derivatives |
|--|---------|-----------|
| 有没有实物？ / Do you own the asset? | 有，你真的拥有它 / Yes, you really own it | 没有，只交易"价格差" / No, you only trade the price difference |
| 要全款吗？ / Full payment required? | 是的，多少钱买多少 / Yes — you pay what it costs | 不用，可以用"杠杆"以小博大 / No — leverage lets you go big with a little |
| 能不能赌跌？ / Can you bet on a fall? | 一般不能（只能买涨）/ Usually no (you can only buy) | 可以！做空就是赌跌 / Yes — shorting bets on a fall |
| 风险大吗？ / Is the risk big? | 相对小（最多亏光本金）/ Relatively small (loss is capped at capital) | 可以很大（杠杆放大亏损）/ Can be huge — leverage magnifies losses |
| 举个例子 / Examples | 买股票、买比特币现货 / Buying stocks, buying spot Bitcoin | 外汇差价合约(CFD)、期权、期货 / FX CFDs, options, futures |

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, ToggleButtons
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

# Color palette
C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 1.3 互动实验：杠杆到底有多猛？ / 1.3 Interactive Experiment: Just How Fierce Is Leverage?

下面的滑块让你亲手体验：

The sliders below let you feel it first-hand:

- **投入资金**：你口袋里有多少钱 / **Capital** — how much cash is in your pocket
- **杠杆倍数**：平台"借"给你多少倍（1倍=没有杠杆）/ **Leverage ratio** — how much the broker lets you multiply (1× = no leverage)
- **价格变动**：市场涨跌了百分之几 / **Price change** — how much the market moved in percent

> 拖动滑块看看：同样的钱，有杠杆和没杠杆，结果差多少？
>
> Drag the sliders: with the same capital, how different are the outcomes with vs. without leverage?

In [ ]:
def leverage_experiment(capital=1000, leverage=100, price_change=5.0):
    plt.close('all')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # --- Left: P&L curves ---
    change_range = np.linspace(-10, 10, 200)
    spot_pnl = capital * change_range / 100
    lev_pnl = capital * leverage * change_range / 100

    ax = axes[0]
    ax.plot(change_range, spot_pnl, color=C_BLUE, lw=2.5, label='No leverage (1:1)')
    ax.plot(change_range, lev_pnl, color=C_RED, lw=2.5, label=f'Leveraged (1:{leverage})')
    ax.axhline(y=0, color='gray', ls='--', alpha=0.5)
    ax.axhline(y=capital, color=C_RED, ls=':', alpha=0.4, label=f'Your capital ${capital}')
    ax.axhline(y=-capital, color=C_RED, ls=':', alpha=0.4)
    ax.axvline(x=price_change, color=C_ORANGE, ls='--', alpha=0.7, lw=2)
    ax.set_xlabel('Price Change (%)', fontsize=11)
    ax.set_ylabel('P&L (USD)', fontsize=11)
    ax.set_title('With vs Without Leverage', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

    # --- Right: current result bar chart ---
    ax = axes[1]
    spot_result = capital * price_change / 100
    lev_result = capital * leverage * price_change / 100
    bar_colors = [C_GREEN if spot_result >= 0 else C_RED, C_GREEN if lev_result >= 0 else C_RED]
    bars = ax.bar(['No Lev', f'Lev 1:{leverage}'], [spot_result, lev_result],
                  color=bar_colors, edgecolor='white', lw=2)
    for bar, val, roi in zip(bars, [spot_result, lev_result], [price_change, price_change * leverage]):
        y_pos = bar.get_height() + (80 if val >= 0 else -200)
        ax.text(bar.get_x() + bar.get_width() / 2, y_pos,
                f'P&L: ${val:,.0f}\nROI: {roi:.0f}%', ha='center', fontsize=10, fontweight='bold')
    ax.axhline(y=0, color='gray', ls='--')
    ax.set_ylabel('P&L (USD)', fontsize=11)
    ax.set_title(f'Result at price change {price_change:+.1f}%', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Text summary
    if price_change >= 0:
        print(f'  Price up {price_change}%:')
        print(f'    No leverage: +${capital * price_change / 100:,.0f}')
        print(f'    Leveraged:   +${capital * leverage * price_change / 100:,.0f}')
        print(f'    Leverage amplifies gains {leverage}x!')
    else:
        print(f'  Price down {abs(price_change)}%:')
        print(f'    No leverage: -${abs(capital * price_change / 100):,.0f}')
        print(f'    Leveraged:   -${abs(capital * leverage * price_change / 100):,.0f}')
        if abs(price_change * leverage) >= 100:
            print('    Loss already exceeds capital -> you would be liquidated!')
        else:
            print(f'    Leverage amplifies losses {leverage}x too. That is the risk.')

interact(leverage_experiment,
         capital=IntSlider(value=1000, min=100, max=50000, step=100, description='Capital($):'),
         leverage=IntSlider(value=100, min=1, max=500, step=10, description='Leverage:'),
         price_change=FloatSlider(value=5.0, min=-20, max=20, step=0.5, description='Change(%):'));

## 1.4 金融衍生品的"家族树" / 1.4 The Family Tree of Financial Derivatives

```
                    金融衍生品 / Financial Derivatives
                   /    |    \
                  /     |     \
              远期    期货/期权   互换
            Forward   Futures   Swap
                      Option
              |        |          |
            (场外)   (交易所)    (场外)
            (OTC)    (Exchange)  (OTC)
              |        |          |
           外汇远期  黄金期货   利率互换
           FX Fwd    Gold Fut   IRS
           CFD合约   BTC期权   货币互换
           CFD       BTC Opt   Currency swap
                     股指期货
                     Equity-index fut
                     永续合约 ← 加密货币特有!
                     Perpetual ← crypto-native!
```

- **远期/期货 / Forwards & Futures**：约定未来某个时间以某个价格买卖（第四章详讲）/ Agree to buy or sell at a fixed price on a future date (details in Chapter 4)
- **期权 / Options**：你花一笔小钱，买一个"权利"（第五章详讲）/ Pay a small premium to buy a "right" (details in Chapter 5)
- **互换 / Swaps**：两个人交换各自的现金流（第十章详讲）/ Two parties swap each other's cash flows (details in Chapter 10)
- **永续合约 / Perpetuals**：加密货币世界发明的，没有到期日的期货（第十一章详讲）/ A crypto-born futures contract with no expiry date (details in Chapter 11)

## 1.5 小结 / 1.5 Chapter Recap

| 关键词 / Term | 一句话解释 / One-line Explanation |
|--------|-----------|
| 现货 / Spot | 花钱买实物，一手交钱一手交货 / Pay cash, receive the physical asset |
| 衍生品 / Derivative | 不买实物，只赌价格涨跌 / No asset delivery — you only bet on the price move |
| 杠杆 / Leverage | 平台借你钱，放大盈亏 / The broker lends you money, magnifying gains and losses |
| 做多 / Long | 赌涨，先买后卖 / Bet on a rise — buy first, sell later |
| 做空 / Short | 赌跌，先卖后买 / Bet on a fall — sell first, buy later |

> **下一章：保证金和爆仓** —— 杠杆交易中最重要的风险控制
>
> **Next chapter: Margin & Liquidation** — the single most important risk-control topic in leveraged trading